# NB4 - COCO val2017 Reference Evaluation

Evaluates the three pretrained models on the official COCO val2017 set (5000
images, no training) and checks the measured numbers against published figures.
This is the correctness check on the whole ultralytics pipeline and the reference
context for the paper; it is not a contribution.

**Kaggle setup:** phone-verified account, internet on. No Roboflow key and no
mounted dataset needed; val2017 is fetched directly. A val-only YAML keeps
`model.val()` from pulling the 19 GB train2017 archive.

**Runtime ~30 to 40 min** (download ~5 min, three evals on 5000 images). Fully
independent of NB1 to NB3, so it fits any free GPU slot.


## 0. One-time environment setup

In [ ]:
# Same ultralytics pin as the other notebooks. Uninstall ray to avoid the raytune callback.
! pip install -q ultralytics==8.3.40 pillow==10.4.0
! pip uninstall -y -q ray 2>/dev/null || true

## 1. Imports, constants, seeds

In [ ]:
# Standard library
import json, os, sys, shutil, subprocess, time, random, zipfile, urllib.request
from pathlib import Path

# Block ultralytics 8.3.40's broken raytune callback
sys.modules['ray'] = None

# Numeric
import numpy as np
import pandas as pd

# Deep learning
import torch

# Reproducibility
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Paths (Kaggle)
KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT   = Path('/kaggle/input')
COCO_DIR       = KAGGLE_WORKING / 'data' / 'coco'
RUNS_DIR       = KAGGLE_WORKING / 'runs'
RESULTS_DIR    = KAGGLE_WORKING / 'results'
STATE_FILE     = KAGGLE_WORKING / 'nb4_state.json'

IMGSZ  = 640
DEVICE = 0

# Pretrained checkpoints (variants stated explicitly)
YOLO_WEIGHTS   = 'yolov8s.pt'
RTDETR_WEIGHTS = 'rtdetr-l.pt'
WORLD_WEIGHTS  = 'yolov8s-worldv2.pt'

# Published COCO val2017 mAP50-95 for the correctness check (ultralytics model cards).
# YOLO-World zero-shot varies with the label set, so it has no fixed reference.
REFERENCE = {'yolov8s': 0.449, 'rtdetr-l': 0.530}
REF_TOL   = 0.03   # measured within this of published -> pipeline configured correctly

COCO_IMAGES_ZIP = 'http://images.cocodataset.org/zips/val2017.zip'
COCO_LABELS_ZIP = 'https://github.com/ultralytics/yolov5/releases/download/v1.0/coco2017labels.zip'

COCO_NAMES = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck','boat','traffic light',
    'fire hydrant','stop sign','parking meter','bench','bird','cat','dog','horse','sheep','cow',
    'elephant','bear','zebra','giraffe','backpack','umbrella','handbag','tie','suitcase','frisbee',
    'skis','snowboard','sports ball','kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket','bottle',
    'wine glass','cup','fork','knife','spoon','bowl','banana','apple','sandwich','orange',
    'broccoli','carrot','hot dog','pizza','donut','cake','chair','couch','potted plant','bed',
    'dining table','toilet','tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear','hair drier','toothbrush',
]

DATASET_SLUG = 'ausdm-v3-coco'
KAGGLE_USERNAME_OVERRIDE = ''   # set if auto-detect returns 'unknown'

for d in [COCO_DIR, RUNS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'KAGGLE_WORKING = {KAGGLE_WORKING}')
print(f'reference      = {REFERENCE} (tol +/-{REF_TOL})')
print(f'classes        = {len(COCO_NAMES)}')
print(f'SEED           = {SEED}')

## 2. Resumable state and restore

Restore from a prior `ausdm-v3-coco` mount so a re-run skips completed evals.


In [ ]:
# _load_state / _mark_done / _is_done - stage-level idempotency across sessions
def _load_state():
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text())
    return {}

def _mark_done(stage, extra=None):
    st = _load_state()
    st[stage] = {'done': True, 'ts': time.time()}
    if extra:
        st[stage].update(extra)
    STATE_FILE.write_text(json.dumps(st, indent=2))

def _is_done(stage):
    return _load_state().get(stage, {}).get('done', False)

# _restore_from_prior_push - pull a prior NB4 output back into working
def _restore_from_prior_push():
    if STATE_FILE.exists():
        return 'local state already present'
    if not KAGGLE_INPUT.exists():
        return 'no /kaggle/input'
    for cand in KAGGLE_INPUT.rglob('nb4_state.json'):
        src = cand.parent
        shutil.copy2(cand, STATE_FILE)
        for f in ['coco_reference_metrics.csv']:
            if (src / f).exists():
                shutil.copy2(src / f, RESULTS_DIR / f)
        return f'restored from {src}'
    return 'no prior push mounted'

print('restore:', _restore_from_prior_push())
print('current state:', json.dumps(_load_state(), indent=2))

## 3. Kaggle username - fail before the push

In [ ]:
# _kaggle_user - try every credential source Kaggle might expose, in order
def _kaggle_user():
    if KAGGLE_USERNAME_OVERRIDE:
        return KAGGLE_USERNAME_OVERRIDE
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        u = api.config_values.get('username') or api.get_config_value('username')
        if u:
            return u
    except Exception:
        pass
    u = os.environ.get('KAGGLE_USERNAME')
    if u:
        return u
    kj = Path.home() / '.kaggle' / 'kaggle.json'
    if kj.exists():
        try:
            return json.loads(kj.read_text())['username']
        except Exception:
            pass
    if KAGGLE_INPUT.exists():
        for p in KAGGLE_INPUT.rglob('*'):
            parts = p.parts
            if 'datasets' in parts:
                i = parts.index('datasets')
                if i + 1 < len(parts):
                    return parts[i + 1]
    return 'unknown'

KAGGLE_USER = _kaggle_user()
print('Kaggle username:', KAGGLE_USER)
if KAGGLE_USER == 'unknown':
    print('WARNING: set KAGGLE_USERNAME_OVERRIDE in section 1 before the dry-run push.')

## 4. Fetch COCO val2017 and write the val-only YAML

Downloads images (~780 MB) and the YOLO-format labels (~46 MB). The YAML points
`train`, `val`, and `test` all at val2017, which is what blocks the 19 GB
train2017 pull (plan section 9).


In [ ]:
# _fetch_coco_val - images + YOLO-format labels, skip if already present
def _fetch_coco_val():
    val_img_dir = COCO_DIR / 'images' / 'val2017'
    val_lab_dir = COCO_DIR / 'labels' / 'val2017'
    if val_img_dir.exists() and len(list(val_img_dir.glob('*.jpg'))) >= 4900 and val_lab_dir.exists():
        return val_img_dir, val_lab_dir
    tmp = KAGGLE_WORKING / 'coco_download'; tmp.mkdir(exist_ok=True)
    imgs_zip = tmp / 'val2017.zip'; labs_zip = tmp / 'coco2017labels.zip'
    if not imgs_zip.exists():
        print('downloading val2017 (~780 MB)...'); urllib.request.urlretrieve(COCO_IMAGES_ZIP, imgs_zip)
    if not labs_zip.exists():
        print('downloading YOLO labels (~46 MB)...'); urllib.request.urlretrieve(COCO_LABELS_ZIP, labs_zip)
    (COCO_DIR / 'images').mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(imgs_zip) as z:
        z.extractall(COCO_DIR / 'images')
    labs_stage = tmp / 'labels_stage'; labs_stage.mkdir(exist_ok=True)
    with zipfile.ZipFile(labs_zip) as z:
        z.extractall(labs_stage)
    src = None
    for cand in labs_stage.rglob('val2017'):
        if cand.is_dir() and any(cand.glob('*.txt')):
            src = cand; break
    assert src is not None, 'Could not find labels/val2017 in the labels zip'
    val_lab_dir.parent.mkdir(parents=True, exist_ok=True)
    if val_lab_dir.exists():
        shutil.rmtree(val_lab_dir)
    shutil.move(str(src), str(val_lab_dir))
    return val_img_dir, val_lab_dir

if not _is_done('coco_fetch'):
    IMG, LAB = _fetch_coco_val()
    n_img = len(list(IMG.glob('*.jpg'))); n_lab = len(list(LAB.glob('*.txt')))
    print(f'val2017 images: {n_img}, labels: {n_lab}')
    assert n_img >= 4900, f'Expected 5000, got {n_img}'
    _mark_done('coco_fetch', {'images': n_img, 'labels': n_lab})
else:
    print('SKIP coco_fetch:', _load_state()['coco_fetch'])

# _write_coco_yaml - train and val both point at val2017
COCO_YAML = COCO_DIR / 'coco_val_only.yaml'
COCO_YAML.write_text(f'''path: {COCO_DIR}
train: images/val2017
val:   images/val2017
test:  images/val2017
nc: 80
names: {COCO_NAMES}
''')
print('coco val-only YAML:', COCO_YAML)

## 5. Neutralise the raytune callback

In [ ]:
# Harmless for val-only, kept for consistency with the other notebooks.
try:
    from ultralytics.utils.callbacks import raytune as _rt
    _rt.on_fit_epoch_end.__code__ = (lambda trainer: None).__code__
    _rt.callbacks = {}
    _rt.tune = None
    print('raytune callback neutralised (code swapped to no-op)')
except Exception as e:
    print('raytune patch skipped:', e)

## 6. Authenticated dry-run push

In [ ]:
PUSH_DIR = KAGGLE_WORKING / 'push_stage'

# _stage_push_dir - rebuild staging folder and metadata
def _stage_push_dir(files=None, dirs=None):
    if PUSH_DIR.exists():
        shutil.rmtree(PUSH_DIR)
    PUSH_DIR.mkdir(parents=True, exist_ok=True)
    for f in (files or []):
        if Path(f).exists():
            shutil.copy2(f, PUSH_DIR / Path(f).name)
    for d in (dirs or []):
        if Path(d).exists():
            shutil.copytree(d, PUSH_DIR / Path(d).name, dirs_exist_ok=True)
    meta = {'title': 'AusDM v3 COCO reference',
            'id': f'{KAGGLE_USER}/{DATASET_SLUG}',
            'licenses': [{'name': 'CC-BY-SA-4.0'}]}
    (PUSH_DIR / 'dataset-metadata.json').write_text(json.dumps(meta, indent=2))
    return meta

# _push - create on first run, version-bump thereafter
def _push(message):
    # Robust push. The kaggle CLI can exit 0 while printing 'already in use',
    # so success is judged from the output text, not the return code. If the
    # dataset already exists, version-bump instead of create.
    def _run(sub):
        r = subprocess.run(['kaggle','datasets'] + sub, capture_output=True, text=True)
        return (r.stdout + r.stderr)
    out = _run(['create','-p',str(PUSH_DIR),'-r','zip'])
    if 'invalid owner id' in out.lower():
        print('CREATE:', out.strip()[-200:]); return False
    if 'is being created' in out.lower() or 'successfully' in out.lower():
        print('CREATE ok'); return True
    vout = _run(['version','-p',str(PUSH_DIR),'-m',message,'-r','zip'])
    print('VERSION:', vout.strip()[-220:])
    return ('is being created' in vout.lower()) or ('successfully' in vout.lower())
assert KAGGLE_USER != 'unknown', 'Set KAGGLE_USERNAME_OVERRIDE in section 1 and re-run.'

(RESULTS_DIR / 'nb4_marker.txt').write_text('nb4 dry-run ok\n')
_stage_push_dir(files=[RESULTS_DIR / 'nb4_marker.txt'])
if _push('NB4 dry-run'):
    print('dry-run push OK ->', f'{KAGGLE_USER}/{DATASET_SLUG}')
else:
    raise RuntimeError('Dry-run push failed. Fix auth / username before the evals.')

## 7. Evaluate YOLOv8s (pretrained, 5000 images)

In [ ]:
# _eval_coco - val-split metrics for a pretrained checkpoint on val2017
def _eval_coco(kind, weights, name, classes=None):
    if kind == 'yolo':
        from ultralytics import YOLO; m = YOLO(weights)
    elif kind == 'rtdetr':
        from ultralytics import RTDETR; m = RTDETR(weights)
    else:
        from ultralytics import YOLOWorld; m = YOLOWorld(weights); m.set_classes(classes)
    mt = m.val(data=str(COCO_YAML), split='val', imgsz=IMGSZ, device=DEVICE,
               project=str(RUNS_DIR), name=name, exist_ok=True, verbose=False, plots=False)
    return {'mAP50-95': round(float(mt.box.map),4), 'mAP50': round(float(mt.box.map50),4),
            'precision': round(float(mt.box.mp),4), 'recall': round(float(mt.box.mr),4)}

if not _is_done('eval_yolov8s'):
    r = _eval_coco('yolo', YOLO_WEIGHTS, 'coco_yolov8s')
    r.update({'model':'yolov8s', 'mode':'pretrained'})
    print('YOLOv8s:', json.dumps(r, indent=2))
    _mark_done('eval_yolov8s', {'metrics': r})
else:
    print('SKIP eval_yolov8s:', _load_state()['eval_yolov8s']['metrics'])

## 8. Evaluate RT-DETR-l (pretrained, 5000 images)

In [ ]:
if not _is_done('eval_rtdetr'):
    r = _eval_coco('rtdetr', RTDETR_WEIGHTS, 'coco_rtdetr')
    r.update({'model':'rtdetr-l', 'mode':'pretrained'})
    print('RT-DETR-l:', json.dumps(r, indent=2))
    _mark_done('eval_rtdetr', {'metrics': r})
else:
    print('SKIP eval_rtdetr:', _load_state()['eval_rtdetr']['metrics'])

## 9. Evaluate YOLO-World zero-shot (80 COCO prompts, 5000 images)

The pretrained `yolov8s-worldv2.pt` with the 80 COCO class names set as its
vocabulary. No fixed published reference, so this row is reported, not gated.


In [ ]:
if not _is_done('eval_world'):
    r = _eval_coco('world', WORLD_WEIGHTS, 'coco_world', classes=COCO_NAMES)
    r.update({'model':'yolo-world-zs', 'mode':'zero-shot'})
    print('YOLO-World zero-shot:', json.dumps(r, indent=2))
    _mark_done('eval_world', {'metrics': r})
else:
    print('SKIP eval_world:', _load_state()['eval_world']['metrics'])

## 10. Consolidated metrics and correctness check

In [ ]:
rows = [_load_state()['eval_yolov8s']['metrics'],
        _load_state()['eval_rtdetr']['metrics'],
        _load_state()['eval_world']['metrics']]
df = pd.DataFrame(rows)[['model','mode','mAP50-95','mAP50','precision','recall']]

# attach published reference and delta where one exists
df['published'] = df['model'].map(REFERENCE)
df['delta'] = (df['mAP50-95'] - df['published']).round(4)
df.to_csv(RESULTS_DIR / 'coco_reference_metrics.csv', index=False)
print(df.to_string(index=False))

checks = []
for _, row in df.iterrows():
    if pd.notna(row['published']):
        ok = abs(row['mAP50-95'] - row['published']) <= REF_TOL
        checks.append(ok)
        print(f'{row["model"]:14s} measured {row["mAP50-95"]:.4f}  vs published {row["published"]:.3f}  '
              f'-> {"OK" if ok else "OFF"}')
pipeline_ok = bool(all(checks)) if checks else False
print('\npipeline correctness:', 'PASS' if pipeline_ok else 'CHECK')
_mark_done('consolidated', {'pipeline_ok': pipeline_ok})

**10.1 Explanation.**

- YOLOv8s and RT-DETR-l should land within about 0.03 of their published COCO
  val2017 mAP50-95 (0.449 and 0.530). That agreement is the evidence the whole
  ultralytics evaluation path is configured correctly.
- This is the free correctness check promised in the plan (section 2.1): it also
  shows the v2 COCO numbers, which sat far above published state of the art, were
  not a standard evaluation.
- YOLO-World zero-shot has no single published value; report it as measured.


## 11. Final push - `ausdm-v3-coco`

In [ ]:
_mark_done('final_push')
shutil.copy2(STATE_FILE, RESULTS_DIR / 'nb4_state.json')
_stage_push_dir(dirs=[RESULTS_DIR])
print('final push:', 'OK' if _push('NB4 final: COCO reference') else 'FAILED')

## 12. Conclusion and handoff to NB5

In [ ]:
print('NB4 state:', json.dumps(_load_state(), indent=2))
print()
print('=== COCO val2017 reference metrics ===')
print(pd.read_csv(RESULTS_DIR / 'coco_reference_metrics.csv').to_string(index=False))
print()
print('pipeline correctness:', 'PASS' if _load_state().get('consolidated',{}).get('pipeline_ok') else 'CHECK')
print('Pushed :', f'{KAGGLE_USER}/{DATASET_SLUG}')
print('Next   : NB5 mounts this for the reference row of Table 1')

**Result.** The three pretrained models are evaluated once on the full COCO
val2017 set with no training. YOLOv8s and RT-DETR-l are compared to their
published mAP50-95 as a correctness check on the ultralytics pipeline; YOLO-World
zero-shot is reported as measured. The metrics CSV is pushed as `ausdm-v3-coco`
for NB5.

**Role in the paper.** Reference context, not a contribution. Agreement with
published COCO figures is the quiet evidence that the container and ICDAR numbers
come from a correctly configured pipeline, and it is why the v3 COCO results
should not be compared to the inflated v2 table.
